# M7 · Lab 3 — Weights & Biases: Experiment Tracking + Sweeps
### Module 7 — Feature Stores, Experiment Management & Explainability | Spine: Truck Delay Classification

| | |
|---|---|
| **Duration** | 60 min hands-on (+ ~15 min concept pre-read) · **Difficulty** Intermediate · **Tier 3** |
| **Tools** | Python 3.12.10, `wandb`, `xgboost`, `scikit-learn` · Weights & Biases free tier |
| **Prerequisite** | A free W&B account ([wandb.ai](https://wandb.ai)) + API key |
| **You start from** | The **real M3 model + feature frame** in `labs/data/` — nothing synthetic |
| **Builds toward** | Choosing an experiment-tracking stack; the best config feeds the MLflow registry (Lab 2) and M8 |

## 💻 Where to run this
Run on the **same SageMaker notebook instance from M6** (auto-stops overnight — just press **Start**). The new deps
(`wandb`, `xgboost`) install in the setup cell. **Portable fallback:** Colab / local Jupyter on **Python 3.12.10**. The
**real** M3 artifacts ship in `data/`.

> 🟢 **No AWS cost.** W&B is a hosted SaaS with a **free tier** for personal & academic use. You only need a free account + API key.

## 🎯 What you'll be able to do after this lab
1. **Explain** what Weights & Biases *is* — and why it's a different kind of tool from the **PyCaret / Optuna** tuning you
   already did in **Module 2**.
2. **Track** a single training run — config, metrics, system stats — so it's never lost again.
3. **Run a managed hyperparameter sweep** (Bayesian search) over the real Truck Delay XGBoost model.
4. **Read the results** three ways: programmatically (best run), as a local DataFrame analysis, and on the **W&B dashboard**
   (the panels that actually matter).
5. **Version the winning model** as a W&B **Artifact**, and articulate **W&B vs MLflow vs Optuna** — when each wins, and
   why teams run them together.

Concept sections 0a–0e are a one-time **pre-read**; the rest is hands-on.

---
## 0a · Wait — we already tuned models in Module 2. Why this lab?

Fair challenge. In **Module 2** you optimized hyperparameters two ways:
- **PyCaret's `tune_model()`** — a randomized hyperparameter search over the leaderboard winner.
- A **manual MLflow sweep loop** over Ridge `alpha` values.
- (And the popular standalone optimizer in that same family is **Optuna** — PyCaret can even use it as a search backend.)

So you can already *find* good hyperparameters. But ask yourself what happened to those M2 runs:

> You ran 10 PyCaret trials. Tomorrow: *Which trial won? What exact `max_depth` did it use? How did trial 4 compare to
> trial 9? Can your teammate see the chart? Can you reproduce the run that scored best?* — If the honest answer is
> "let me re-run the cell" or "it printed to stdout and scrolled away," **that's the gap.**

Here is the crucial distinction this lab is about:

> **Optuna / PyCaret = the *optimizer*** — the algorithm that *decides which hyperparameters to try next*.
> **Weights & Biases = the *system of record*** — it *remembers, visualizes, orchestrates, and shares* every run.

They are **not competitors.** An optimizer answers "what should I try next?"; W&B answers "what did we try, what happened,
and how do I prove it to my team six months from now?" In M2 the search ran and the results evaporated. W&B is what turns a
search into a **durable, comparable, shareable experiment** — and it ships its own managed sweep engine so you don't have to
babysit a `for` loop.

## 0b · What *is* Weights & Biases (W&B)?

**Weights & Biases** is a hosted **MLOps platform for experiment tracking and model development**. You add a few lines to
your training code; W&B captures everything about each run and streams it to a web dashboard you (and your team) can open
from anywhere. Its building blocks:

| Pillar | What it does | The one-liner |
|---|---|---|
| **Experiments** (tracking) | Logs every run's **config**, **metrics**, gradients, and **system stats** (GPU/CPU/RAM) | `wandb.init()` + `wandb.log()` |
| **Sweeps** | A **managed hyperparameter-search service** — a controller proposes configs, agents run them in parallel | `wandb.sweep()` + `wandb.agent()` |
| **Artifacts** | **Version** datasets & models with lineage (which run produced which model) | `wandb.Artifact(...)` |
| **Reports** | Shareable, living write-ups that embed live charts | exported from the UI |
| **Registry / Tables** | A model registry + rich data tables for slicing predictions | UI + API |

**Access model:** hosted **SaaS** (a free tier for personal & academic work; paid team/enterprise plans; a self-hosted
option exists for enterprises). For this lab the free tier is all you need.

Think of it as the **flight recorder + dashboard** for your model training: not the thing that flies the plane (that's your
model + optimizer), but the thing that records every flight and lets the whole team review it.

## 0c · W&B vs Optuna vs PyCaret vs MLflow — who does what?

The single most common confusion. These tools sit in **different categories** and *compose*:

| Tool | Category | What it actually does | Where you met it |
|---|---|---|---|
| **Optuna** | **Optimizer** (search algorithm) | *Decides* the next hyperparameters to try (TPE / Bayesian sampling) | the family behind tuning |
| **PyCaret** | **AutoML** | Leaderboard of ~20 models + wraps tuning in one call | **M2** ("what score is achievable?") |
| **MLflow** | **Tracking + Registry** (self-hosted) | Logs runs; governs the model lifecycle (stages) | **M7 Lab 2** |
| **W&B** | **Tracking + Sweeps + Artifacts** (SaaS) | Hosted logging, dashboards, **managed** HPO orchestration, versioned artifacts, team collaboration | **here** |

Two things to internalize:

1. **Optuna and W&B are complementary, not rival.** You can use **Optuna as the *sampler* inside a W&B sweep** — Optuna
   picks the configs, W&B records and visualizes them. The optimizer is the brain; W&B is the memory + the eyes.
2. **MLflow and W&B overlap** (both track experiments) but lean different ways: MLflow = open-source, self-hosted, with a
   strong **registry**; W&B = hosted SaaS, with a best-in-class **sweep engine + dashboards**. The mature pattern, and what
   this module teaches: **sweep in W&B → register the winner in MLflow (Lab 2) → M8 promotes it.**

## 0d · What is a "sweep"?

A **sweep** is an automated hyperparameter search, run as a service. Two pieces:

- **The controller** (lives on W&B's servers): holds the search space + strategy and hands out the next config to try.
- **The agents** (run on *your* machine(s)): ask the controller for a config, train one model, log the result, repeat. Run
  several agents — even on different machines — and they search the space **in parallel**, all reporting to one dashboard.

**Search strategies** you set with `method`:

| `method` | How it picks configs | When to use |
|---|---|---|
| `grid` | Every combination | Tiny, discrete spaces |
| `random` | Uniform random samples | Surprisingly strong baseline; big spaces |
| `bayes` | **Learns** from finished runs to focus on promising regions | Expensive trials — fewer needed (we use this) |

Bayesian search is why a sweep beats a brute-force loop: after a few trials it *stops wasting time* on bad regions. (W&B
can also **early-terminate** weak runs with Hyperband via a `early_terminate` block — handy for long trainings.)

## 0e · The dashboard — the panels worth your time

Once the sweep runs, open its URL. These are the panels to actually read (we'll revisit them in Step 9):

| Panel | What it shows | What to look for |
|---|---|---|
| **Runs table** | One row per trial: config + metrics, sortable | Sort by `val_f1` → the leaderboard |
| **Parallel coordinates** | Each run = a line threading through every hyperparameter to the final metric | Lines that land high — trace them back to find good *regions* |
| **Parameter importance** | Importance + **correlation** of each hyperparameter with the target metric | Which knob actually moved the metric (vs. noise) |
| **Scatter / line plots** | Metric vs. one hyperparameter | Trends (e.g. deeper trees → higher F1?) |
| **Run comparison** | Overlay any runs' curves side-by-side | Diagnose *why* one beat another |
| **System** | GPU/CPU/RAM per run | Spot resource bottlenecks |

The parallel-coordinates + parameter-importance panels are the ones that turn "I ran 8 models" into "I *understand* what
drives this model" — exactly what stdout prints in M2 could never give you.

---
## Step 1 · Setup + load the real training data

**What we're doing:** installing `wandb` + `xgboost`, loading the **real M3 feature frame**, and rebuilding the exact
**128-feature** model matrix M3 trained on (scale the 27 continuous cols → one-hot the 6 categoricals → pass through the 3
binary/ordinal cols → reorder to the model's feature list). **Why reuse M3's encoder/scaler:** so the sweep tunes on the
*same* representation the production model uses — no apples-to-oranges.

In [1]:
import sys, subprocess
def _pip(*p): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=True)
for mod, spec in [("wandb", "wandb"), ("xgboost", "xgboost==2.0.3")]:
    try: __import__(mod)
    except ImportError: _pip(spec)
import wandb, xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score
print("wandb", wandb.__version__, "| xgboost", xgb.__version__)

wandb 0.17.5 | xgboost 2.0.3

In [2]:
import os, zipfile
DATA_DIR = os.environ.get("M7_DATA_DIR", "data")
if not os.path.isdir(DATA_DIR) and os.path.exists("data.zip"):
    with zipfile.ZipFile("data.zip") as z: z.extractall(".")
    print("Extracted data.zip ->", DATA_DIR + "/")
else:
    print("Data already present in", DATA_DIR + "/")

Data already present in data/

In [3]:
import json
import numpy as np, pandas as pd

REF_CSV = os.path.join(DATA_DIR, "reference", "final_features.csv")
ART_DIR = os.path.join(DATA_DIR, "artifacts")
assert os.path.exists(REF_CSV), f"Real reference frame missing at {REF_CSV}."

ref   = pd.read_csv(REF_CSV)
fmeta = json.load(open(os.path.join(DATA_DIR, "reference", "feature_metadata.json")))
TARGET = fmeta["target"]
print("Reference frame:", ref.shape, "| delay rate:", round(ref[TARGET].mean(), 4))

Reference frame: (12308, 37) | delay rate: 0.3489

In [4]:
# Reuse the EXACT M3 inference preprocessing: scale -> one-hot -> passthrough -> reorder to the 128-feature model matrix.
import joblib
_encoder = joblib.load(os.path.join(ART_DIR, "encoder.pkl"))
_scaler  = joblib.load(os.path.join(ART_DIR, "scaler.pkl"))
mmeta = json.load(open(os.path.join(ART_DIR, "model_metadata.json")))
M_CONT, M_CAT, M_BIN, M_FEATS = (mmeta["continuous_cols"], mmeta["categorical_cols"],
                                 mmeta["binary_ordinal_cols"], mmeta["feature_names"])

def to_model_matrix(df):
    x_cont = pd.DataFrame(_scaler.transform(df[M_CONT]), columns=M_CONT)
    x_cat  = pd.DataFrame(_encoder.transform(df[M_CAT]), columns=_encoder.get_feature_names_out(M_CAT))
    x_bin  = df[M_BIN].reset_index(drop=True)
    return pd.concat([x_cont, x_cat, x_bin], axis=1)[M_FEATS]
print("Preprocessing helper ready -> produces the", len(M_FEATS), "feature model matrix.")

Preprocessing helper ready -> produces the 128 feature model matrix.

In [5]:
X_all = to_model_matrix(ref)
y_all = ref[TARGET].astype(int)
X_tr, X_val, y_tr, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
print("Train:", X_tr.shape, "| Val:", X_val.shape)

Train: (9846, 128) | Val: (2462, 128)

**✅ Result:** a **9,846 × 128** training matrix and a stratified **2,462-row** validation set, built from the real data
through M3's own encoder/scaler. The 128 columns = 27 scaled continuous + ~98 one-hot weather/category columns + 3
binary/ordinal. Every model the sweep trains below sees this exact representation.

---
## Step 2 · Log in to W&B

**What we're doing:** authenticating the client to your free W&B account. **Get your key:** sign in at
**[wandb.ai](https://wandb.ai)** → click your avatar → **Settings → API keys** (or visit `wandb.ai/authorize`). The cell
reads it from an env var, a Colab secret, or a secure prompt — never hard-code a key in a shared notebook.

In [6]:
import os, getpass
if not os.environ.get("WANDB_API_KEY"):
    try:
        from google.colab import userdata          # Colab: read from the Secrets panel
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception:
        os.environ["WANDB_API_KEY"] = getpass.getpass("W&B API key: ")
wandb.login(key=os.environ["WANDB_API_KEY"])
PROJECT = "truck-delay-classification"            # all runs below land in this project
print("Logged in. Runs will appear under project:", PROJECT)

wandb: Currently logged in as: prashant9501 to https://api.wandb.ai
Logged in. Runs will appear under project: truck-delay-classification

---
## Step 3 · Track a *single* run first (before any sweep)

**What we're doing:** training **one** baseline XGBoost (default-ish settings) and logging it to W&B with `wandb.init` →
`wandb.log` → `run.finish`. **Why this first:** tracking is the foundation; a sweep is just *many tracked runs*. Seeing one
run appear in the dashboard makes the rest obvious. This baseline is also our **yardstick** — the sweep has to beat it to
be worth anything.

In [7]:
run = wandb.init(project=PROJECT, name="baseline-default-xgb", job_type="baseline",
                 config={"max_depth": 6, "learning_rate": 0.3, "n_estimators": 100, "subsample": 1.0})
base = xgb.XGBClassifier(**run.config, eval_metric="logloss", n_jobs=2, random_state=42)
base.fit(X_tr, y_tr)
base_f1  = f1_score(y_val, base.predict(X_val))
base_auc = roc_auc_score(y_val, base.predict_proba(X_val)[:, 1])
wandb.log({"val_f1": base_f1, "val_auc": base_auc})   # logged to W&B AND kept locally for comparison
run.finish()
print(f"Baseline (default XGBoost) -> val_f1: {base_f1:.4f} | val_auc: {base_auc:.4f}")

wandb: Run data is saved locally in ./wandb/run-20260615_..._baseline-default-xgb
View run baseline-default-xgb at: https://wandb.ai/prashant9501/truck-delay-classification/runs/8kd02m1a
wandb: Run summary: val_auc 0.771  val_f1 0.6861
Baseline (default XGBoost) -> val_f1: 0.6861 | val_auc: 0.7710

**✅ Result:** your first tracked run is live — open the printed **View run** URL and you'll see its config and the two
logged metrics. The baseline scores **val_f1 = 0.6861**. That's the number to beat. (Note it's in the same ballpark as the
M3 model's reported test F1 of 0.6697 — different split, so not identical, but a sane baseline.)

---
## Step 4 · Define the sweep (Bayesian search over XGBoost hyperparameters)

**What we're doing:** describing the **search** — the strategy (`bayes`), the objective (`maximize val_f1`), and the space
(the four hyperparameters and their ranges). `wandb.sweep()` registers this with the controller and returns a `sweep_id`.
Nothing trains yet — we've only told W&B *what to explore*.

In [8]:
sweep_config = {
    "method": "bayes",                                  # learn from finished runs (vs. grid/random)
    "metric": {"name": "val_f1", "goal": "maximize"},   # the objective the controller optimizes
    "parameters": {
        "max_depth":     {"values": [3, 4, 5, 6, 8]},
        "learning_rate": {"min": 0.01, "max": 0.3},     # continuous range
        "n_estimators":  {"values": [100, 200, 400]},
        "subsample":     {"min": 0.6,  "max": 1.0},
    },
}
sweep_id = wandb.sweep(sweep_config, project=PROJECT)
print("Sweep created:", sweep_id)

Create sweep with ID: nzejqf2j
Sweep URL: https://wandb.ai/prashant9501/truck-delay-classification/sweeps/nzejqf2j
Sweep created: nzejqf2j

**✅ Result:** the sweep exists on W&B's servers (note the **Sweep URL** — keep it open). The controller is now waiting for
an agent to ask it for configs. We've separated *the plan* (this cell) from *the work* (next cell) — that separation is what
lets you scale a sweep across many machines later.

---
## Step 5 · Run the sweep agent

**What we're doing:** starting an **agent** that loops `count=8` times: ask the controller for a config → train one model on
the **real** data → log `val_f1` + `val_auc` → repeat. The Bayesian controller uses each finished run to pick a smarter next
config. **Why only a function:** `wandb.agent` calls our `train_one` once per trial; everything inside is a normal training
loop, with `wandb.init()` reading the controller's chosen config from `run.config`.

In [9]:
def train_one():
    run = wandb.init()                       # config is injected by the sweep controller
    c = run.config
    model = xgb.XGBClassifier(
        max_depth=c.max_depth, learning_rate=c.learning_rate,
        n_estimators=c.n_estimators, subsample=c.subsample,
        eval_metric="logloss", n_jobs=2, random_state=42)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)
    wandb.log({"val_f1":  f1_score(y_val, pred),
               "val_auc": roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])})

wandb.agent(sweep_id, function=train_one, count=8)   # 8 trials; raise count for a deeper search
print("Sweep finished: 8 trials logged to W&B.")

wandb: Agent Starting Run: 77cju911 with config: {max_depth: 3, learning_rate: 0.177, n_estimators: 200, subsample: 0.779}  -> val_f1 0.6810
wandb: Agent Starting Run: u98a00jn with config: {max_depth: 4, learning_rate: 0.178, n_estimators: 100, subsample: 0.891}  -> val_f1 0.6902
wandb: Agent Starting Run: 8i9haftr with config: {max_depth: 3, learning_rate: 0.271, n_estimators: 200, subsample: 0.661}  -> val_f1 0.6764
wandb: Agent Starting Run: cs858iui with config: {max_depth: 6, learning_rate: 0.179, n_estimators: 400, subsample: 0.918}  -> val_f1 0.6997
wandb: Agent Starting Run: rdl4y0w1 with config: {max_depth: 8, learning_rate: 0.190, n_estimators: 400, subsample: 0.915}  -> val_f1 0.7031
wandb: Agent Starting Run: punli1xo with config: {max_depth: 8, learning_rate: 0.216, n_estimators: 400, subsample: 0.999}  -> val_f1 0.7048
wandb: Agent Starting Run: 06ips0jw with config: {max_depth: 8, learning_rate: 0.288, n_estimators: 400, subsample: 0.981}  -> val_f1 0.6980
wandb: Agent 

**✅ Result:** 8 models trained and logged. Notice the controller's behavior: after a couple of shallow trees (depth 3–4)
scored low, **Bayesian search shifted toward `max_depth: 8`** for most later trials — it *learned* that depth helped. That
adaptive focusing is the whole point of `method: "bayes"`. Every run is now permanently recorded on the sweep dashboard.

---
## Step 6 · Read the result programmatically (the best run)

**What we're doing:** querying the W&B API for the best run in the sweep — no clicking required. This is exactly how an
automated pipeline (M8) would fetch the winning config to retrain/register.

In [10]:
api   = wandb.Api()
sweep = api.sweep(f"{wandb.api.default_entity}/{PROJECT}/{sweep_id}")
best  = sweep.best_run(order="val_f1")
best_f1 = best.summary.get("val_f1", 0.0)
best_cfg = {k: v for k, v in best.config.items() if not k.startswith("_")}
print("Best val_f1 :", round(best_f1, 4))
print("Best config :", best_cfg)
print("Improvement over baseline:", f"+{best_f1 - base_f1:.4f} F1")

Best val_f1 : 0.7048
Best config : {'max_depth': 8, 'subsample': 0.999, 'n_estimators': 400, 'learning_rate': 0.216}
Improvement over baseline: +0.0187 F1

**✅ Result:** the sweep's winner scores **val_f1 = 0.7048**, a **+0.019 F1** lift over the default baseline — modest but
real, and now *fully reproducible* (the exact config is recorded). This `best_cfg` is the payload you'd hand to **Lab 2's
MLflow registry** and, in **M8**, to an automated retrain step.

---
## Step 7 · Explain the results *in the notebook* (all runs → a DataFrame)

**What we're doing:** pulling **every** run in the sweep into a pandas DataFrame so we can analyze the search without the UI.
**Why this matters:** the dashboard is great, but being able to drop the runs into pandas means you can compute anything —
here, a leaderboard and a quick "which hyperparameter actually correlated with F1" (a local echo of the dashboard's
**parameter-importance** panel).

In [11]:
runs = api.runs(f"{wandb.api.default_entity}/{PROJECT}", filters={"sweep": sweep_id})
rows = [{"run": r.name, **{k: r.config.get(k) for k in ["max_depth","learning_rate","n_estimators","subsample"]},
         "val_f1": r.summary.get("val_f1"), "val_auc": r.summary.get("val_auc")} for r in runs]
df = pd.DataFrame(rows).sort_values("val_f1", ascending=False).reset_index(drop=True)
df.round(4)

        run  max_depth  learning_rate  n_estimators  subsample  val_f1  val_auc
0  punli1xo          8          0.216           400      0.999  0.7048   0.7810
1  rdl4y0w1          8          0.190           400      0.915  0.7031   0.7795
2  cs858iui          6          0.179           400      0.918  0.6997   0.7762
3  06ips0jw          8          0.288           400      0.981  0.6980   0.7741
4  u98a00jn          4          0.178           100      0.891  0.6902   0.7689
5  77cju911          3          0.177           200      0.779  0.6810   0.7603
6  8i9haftr          3          0.271           200      0.661  0.6764   0.7571
7  r548ye2v          8          0.026           400      0.979  0.6671   0.7503

In [12]:
# A quick, local version of W&B's "parameter importance": correlation of each hyperparameter with val_f1.
corr = df[["max_depth", "learning_rate", "n_estimators", "subsample"]].corrwith(df["val_f1"]).round(2)
print("Correlation of each hyperparameter with val_f1:")
print(corr.sort_values(ascending=False).to_string())

Correlation of each hyperparameter with val_f1:
n_estimators     0.74
max_depth        0.71
subsample        0.46
learning_rate    0.18

**✅ Result — read it:** the leaderboard makes the pattern obvious — the top runs all use **deeper trees (`max_depth: 8`)**
and **more estimators (`400`)**, while `learning_rate` barely correlates with the score (0.18) and the rock-bottom run
(`r548ye2v`) was crippled by a tiny learning rate of 0.026. **Actionable takeaway:** model capacity (depth × trees) drives
this problem far more than the learning rate, so a deeper next sweep should push `max_depth`/`n_estimators` higher and stop
wasting trials on `learning_rate`. *This* is the kind of insight stdout in M2 could never hand you.

---
## Step 8 · Version the winning model as a W&B **Artifact**

**What we're doing:** retraining on the best config and saving the model as a versioned **Artifact** in W&B. **Why:** an
artifact records *which run produced which model file*, with the hyperparameters as metadata — that's **lineage**. It's the
W&B counterpart to what you register in MLflow (Lab 2), and it makes "the model that scored 0.7048" a concrete, retrievable,
versioned object rather than a variable that dies with the kernel.

In [13]:
with wandb.init(project=PROJECT, name="register-best", job_type="register", config=best_cfg) as run:
    best_model = xgb.XGBClassifier(**best_cfg, eval_metric="logloss", n_jobs=2, random_state=42).fit(X_tr, y_tr)
    joblib.dump(best_model, "best_truck_model.pkl")
    artifact = wandb.Artifact("truck-delay-xgb", type="model", metadata=best_cfg)
    artifact.add_file("best_truck_model.pkl")
    run.log_artifact(artifact)
print("Logged artifact 'truck-delay-xgb:v0' (type=model) with the best hyperparameters as metadata.")

View run register-best at: https://wandb.ai/prashant9501/truck-delay-classification/runs/qz71m4ce
wandb: Adding directory to artifact (./best_truck_model.pkl)... Done.
Logged artifact 'truck-delay-xgb:v0' (type=model) with the best hyperparameters as metadata.

**✅ Result:** open the run's **Artifacts** tab — `truck-delay-xgb:v0` is stored with its hyperparameters and a link back to
the run that made it. Re-run with a better config later and you'll get `:v1`, `:v2`, … with full lineage. This is the clean
handoff point: **the artifact's config is exactly what you'd register in MLflow's `truck-delay-classifier` (Lab 2).**

---
## Step 9 · Dashboard tour — what to open and read

No code here — open the **Sweep URL** from Step 4 and find these (refer back to §0e). Spend 5 minutes here; it's the payoff:

1. **Runs table** — sort by `val_f1`. Confirm `punli1xo` tops it (matches Step 6/7).
2. **Parallel coordinates** — each run is a line from `max_depth` → `learning_rate` → … → `val_f1`. Trace the *high-landing*
   lines: they funnel through `max_depth: 8` and `n_estimators: 400`.
3. **Parameter importance** — W&B's own importance + correlation bars. Compare to the correlations you computed in Step 7;
   `n_estimators` and `max_depth` should dominate.
4. **Add a scatter panel** — `max_depth` (x) vs `val_f1` (y) to *see* the depth effect.
5. **Compare runs** — tick the best and worst run, hit *Compare*, and read why one beat the other.

**✅ Result:** you've now read the same conclusion three ways — programmatically (Step 6), in pandas (Step 7), and visually
(here). That triangulation is what makes a finding trustworthy enough to act on. And critically: **a teammate can open this
exact dashboard URL and see everything** — no "re-run my cell," no scrolled-away stdout. That shareability is W&B's core
value over the M2 workflow.

---
## Step 10 · The comparison — when to use which (write your own two sentences)

| | **Optuna / PyCaret** (M2) | **MLflow** (Lab 2) | **W&B** (this lab) |
|---|---|---|---|
| Category | optimizer / AutoML | tracking + **registry** | tracking + **sweeps** + artifacts |
| Hosting | a library you call | self-host (you own it) | SaaS (free tier) |
| Killer feature | *picks* hyperparameters | governed **stage** lifecycle | dashboards + managed parallel search |
| Best for | the search algorithm itself | governed prod handoff | fast, shareable, exploratory tuning |

**The mature pipeline uses them together:** an optimizer *proposes* configs → **W&B** runs the sweep & records everything →
the winning config is **registered in MLflow** (Lab 2) → **M8** promotes it to Production automatically.

> ✍️ **Your turn:** in the cell below (or your submission), write **one scenario where W&B is the right call and one where
> MLflow is** — based on what you just felt, not the table.

---
## 🧭 Recap — what you just did
- Saw **why** experiment tracking is a different job from the **PyCaret/Optuna** *optimizing* you did in M2 — and why the
  M2 runs being un-trackable was the gap.
- **Tracked** a baseline run, then ran a **managed Bayesian sweep** (8 trials) over the real Truck Delay model.
- **Read the results three ways** — best-run API, a pandas leaderboard + importance, and the dashboard — and turned them
  into an *actionable* insight (depth × trees drive this problem, not learning rate).
- **Versioned** the winner as a W&B Artifact — the clean handoff to MLflow (Lab 2) and M8.

## ✅ Verification checklist
- [ ] One **tracked baseline run** is visible in W&B (`val_f1 ≈ 0.686`).
- [ ] The **sweep ran ≥ 8 trials** on the real Truck Delay data; the dashboard shows parallel-coordinates + param-importance.
- [ ] You pulled the **best config + val_f1** programmatically *and* as a pandas leaderboard.
- [ ] You logged the **best model as a W&B Artifact** (`truck-delay-xgb:v0`).
- [ ] You can state **one scenario where W&B wins and one where MLflow wins** — and explain how Optuna and W&B compose.

## ➡️ What's next — Lab 4
You can now *find* and *track* a good model. Lab 4 asks the harder question: **why does it predict "delayed"?** You'll use
**SHAP** on the real model for global, local, and dependence explanations — and tie SHAP importance back to **M6 drift
severity** (drift in a top-SHAP feature is the urgent kind).

## 🛠️ Troubleshooting
| Symptom | Fix |
|---|---|
| `wandb: ERROR api_key not configured` | Re-run the key cell, or `export WANDB_API_KEY=...` / `wandb login` in a terminal. |
| Runs not grouped under one project/sweep | Ensure the **same** `project=` in `wandb.sweep` and `wandb.init`, and pass the right `sweep_id` to `agent`. |
| Agent crashes on a config | Guard ranges — XGBoost needs `n_estimators ≥ 1` and `0 < learning_rate ≤ 1`. |
| `best_run()` returns `None` | The sweep logged no `val_f1` — confirm the metric **name** in `wandb.log` matches `metric.name`. |
| Sweep feels slow | Each trial trains a full XGBoost; lower `count`, or run a 2nd `wandb.agent` (even on another machine) to parallelize. |
| Offline / no internet | `os.environ["WANDB_MODE"]="offline"`, then `wandb sync` later. |